# Calculate summary statistics

In [1]:
import pandas as pd

df_titanic = pd.read_csv("https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv")

### Aggregating statistics

![](../assets/pandas/aggregating_statistics.png)

What is the average age of the Titanic passengers?

In [2]:
df_titanic["Age"].mean()

np.float64(29.69911764705882)

Different statistics are available and can be applied to columns with numerical data. You can use the `describe` for a summary:

In [3]:
df_titanic["Age"].describe()

count    714.000000
mean      29.699118
std       14.526497
min        0.420000
25%       20.125000
50%       28.000000
75%       38.000000
max       80.000000
Name: Age, dtype: float64

Note: Operations in general **exclude missing data**.

#### Statistics of multiple columns

![](../assets/pandas/aggregating_statistics_2.png)

What is the median age and ticket fare price of the Titanic passengers?

In [4]:
df_titanic[["Age", "Fare"]].median()

Age     28.0000
Fare    14.4542
dtype: float64

Or using `describe` for all statistics:

In [5]:
df_titanic[["Age", "Fare"]].describe()

,Age,Fare
count,714.000000,891.000000
mean,29.699118,32.204208
std,14.526497,49.693429
min,0.420000,0.000000
25%,20.125000,7.910400
50%,28.000000,14.454200
75%,38.000000,31.000000
max,80.000000,512.329200


Or select a subset of statistics using `agg` method:

In [6]:
df_titanic[["Age", "Fare"]].agg(
    {
        "Age": ["min", "max", "median", "skew"],
        "Fare": ["min", "max", "median", "mean"],
    }
)

,Age,Fare
min,0.420000,0.000000
max,80.000000,512.329200
median,28.000000,14.454200
skew,0.389108,NaN
mean,NaN,32.204208


## Aggregating statistics grouped by category

![](../assets/pandas/groupby.png)


What is the average age for male versus female Titanic passengers?

In [7]:
agg = df_titanic[["Sex", "Age"]].groupby("Sex").mean()
agg

,Age
Sex,
female,27.915709
male,30.726645


Calculating a given statistic (e.g. `mean` age) _for each category in a column_ (e.g. male/female in the `Sex` column) is a common pattern. The `groupby` method is used to support this type of operations. This fits in the more general `split-apply-combine` pattern:

1. **Split** the data into groups
2. **Apply** a function to each group independently
3. **Combine** the results into a data structure
    
The apply and combine steps are typically done together in pandas.


Notice how in this case, the `index` lists the two values: `female` and `male`:

In [8]:
agg.index

Index(['female', 'male'], dtype='str', name='Sex')

And the columns outputs `Age`:

In [9]:
agg.columns

Index(['Age'], dtype='str')

This means you can access using the `loc` notation for label-based access, as follows:

In [10]:
agg.loc['male', 'Age']

np.float64(30.72664459161148)

In the previous example, we explicitly selected the 2 columns first. If not, the `mean` method is applied to each column containing numerical columns by passing `numeric_only=True`:

In [11]:
df_titanic.groupby("Sex").mean(numeric_only=True)

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
Sex,,,,,,,
female,431.028662,0.742038,2.159236,27.915709,0.694268,0.649682,44.479818
male,454.147314,0.188908,2.389948,30.726645,0.429809,0.235702,25.523893


It does not make much sense to get the average value of the `Pclass`. If we are only interested in the average age for each gender, the selection of columns (square brackets `[]` as usual) is supported on the grouped data as well:

In [12]:
df_titanic.groupby("Sex")["Age"].mean()

Sex
female    27.915709
male      30.726645
Name: Age, dtype: float64

![](../assets/pandas/aggregating_statistics_3.png)

Grouping can be done by multiple columns at the same time. Provide the column names as a list to the [`groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html#pandas.DataFrame.groupby "pandas.DataFrame.groupby") method.

We can ask ask a more complex question like: What is the mean ticket fare price for each of the sex and cabin class combinations?

In [13]:
df_titanic.groupby(["Sex", "Pclass"])["Fare"].mean()

Sex     Pclass
female  1         106.125798
        2          21.970121
        3          16.118810
male    1          67.226127
        2          19.741782
        3          12.661633
Name: Fare, dtype: float64

## Count number of records by category

![](../assets/pandas/agg_counts.png)

What is the number of passengers in each of the cabin classes?

In [14]:
df_titanic["Pclass"].value_counts()

Pclass
3    491
1    216
2    184
Name: count, dtype: int64

The [`value_counts()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html#pandas.Series.value_counts "pandas.Series.value_counts") method counts the number of records for each category in a column.

The function is a shortcut, it is actually a groupby operation in combination with counting the number of records within each group:

In [15]:
df_titanic.groupby("Pclass")["Pclass"].count()

Pclass
1    216
2    184
3    491
Name: Pclass, dtype: int64

---

References:

- https://pandas.pydata.org/docs/getting_started/intro_tutorials/06_calculate_statistics.html